# Simulation Experimental Table generator

This code generates the input space for the dataset for model training

The parameters to vary for the simulation are thermal conductivity and boundary temperatures.

Thermal conductivity is the most influencing material parameter on the casting (3% variability) and die temperature at the boundaries is the process parameters.(25% variability).

In [51]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import json

try:
    current_dir = os.getcwd()
except:
    current_dir = os.path.dirname(os.path.abspath(__file__))
    
training_data_dir = os.path.join(current_dir, '..')

sys.path.insert(0, str(training_data_dir))

from simdata_mush_dirc_icc import *


In [52]:
# Data Prep
settings_path_1 = os.path.join(current_dir, '..', 'settings.json')
with open(settings_path_1,'r') as file:
    settings = json.load(file)


heat_data = HT_sim(settings)
alpha = heat_data.alpha_l
tempfield = heat_data.datagen()


In [53]:
from scipy.stats.qmc import Sobol
import numpy as np

# Your parameter ranges as a dictionary

L_f_nominal = 397.5e3
L_f_min =L_f_nominal - 0.03 * L_f_nominal
L_f_max = L_f_nominal + 0.03 * L_f_nominal
die_temp_l_nominal = 820.0
die_temp_r_nominal = 820.0
die_temp_l_min = die_temp_l_nominal - 0.25 * die_temp_l_nominal
die_temp_l_max = die_temp_l_nominal + 0.25 * die_temp_l_nominal
die_temp_r_min = die_temp_r_nominal - 0.25 * die_temp_r_nominal
die_temp_r_max = die_temp_r_nominal + 0.25 * die_temp_r_nominal

print(f"L_f range: {L_f_min:.2e} to {L_f_max:.2e}")
print(f"Die temp left range: {die_temp_l_min:.2f} to {die_temp_l_max:.2f}")
print(f"Die temp right range: {die_temp_r_min:.2f} to {die_temp_r_max:.2f}")

param_ranges = {
    'x': (0.0, 15.0e-3),
    't': (0.0, 7.0),
    'L_fusion': (L_f_min, L_f_max),
    'die_temp_l': (die_temp_l_min, die_temp_l_max),
    'die_temp_r': (die_temp_r_min, die_temp_r_max)
}

n_samples = int(1e5)
keys = list(param_ranges.keys())
bounds = list(param_ranges.values())
n_params = len(bounds)

# Generate Sobol samples in [0, 1)
sobol = Sobol(d=n_params, scramble=True)
samples_unit = sobol.random(n=n_samples)

# Scale samples to actual ranges
samples_scaled = np.array([
    samples_unit[:, i] * (high - low) + low
    for i, (low, high) in enumerate(bounds)
]).T

# Convert to dict for readability
samples_dict = {key: samples_scaled[:, i] for i, key in enumerate(keys)}

# Optional: Convert to structured array or dataframe
import pandas as pd
df_samples = pd.DataFrame(samples_dict)

print(df_samples.shape)

L_f range: 3.86e+05 to 4.09e+05
Die temp left range: 615.00 to 1025.00
Die temp right range: 615.00 to 1025.00
(100000, 5)


/opt/anaconda3/envs/pinn/lib/python3.13/site-packages/scipy/stats/_qmc.py:993: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)


In [54]:
# Function to collect the temperature at a specific point
# take a basic json file and then append the key values based on the input 
# return temperatur at the point as per x,t

def temp_cal(settings, df_samples):
    
    temp_at_points = []
    for index, row in df_samples.iterrows():
        x = row['x']
        t = row['t']
        L_fusion = row['L_fusion']
        die_temp_l = row['die_temp_l']
        die_temp_r = row['die_temp_r']
        
        # Update settings with the current sample values
        settings['L_fusion'] = float(L_fusion)
        settings['die_temp_l'] = float(die_temp_l)
        settings['die_temp_r'] = float(die_temp_r)

        # print(settings)
        # Create a new instance of HT_sim with updated settings
        ht_data = HT_sim(settings)
        
        # Get the temperature at the specified point (x, t)
        temp_at_point = temp_at_pt(ht_data, x, t)

        temp_at_points.append(temp_at_point)

    return np.array(temp_at_points)

In [ ]:
temp_data = temp_cal(settings, df_samples)
# time taken for dataset generation is 1326m, 23.5s


In [56]:
# make combined dataframe

dataset = pd.DataFrame({
    'x': df_samples['x'],
    't': df_samples['t'],
    'L_fusion': df_samples['L_fusion'],
    'die_temp_l': df_samples['die_temp_l'],
    'die_temp_r': df_samples['die_temp_r'],
    'temp': temp_data
})
print(dataset.shape)

# convert Dataset to csv file
dataset.to_csv(os.path.join(current_dir, 'simulated_dataset.csv'), index=False)


(100000, 6)


In [57]:
# evaluate the dataset

